# Phase 6 · Architecture 1: LegalRoPE · Abstractive legal summarization

In [1]:
from pathlib import Path
import os, json, math, random, sys, pickle, subprocess, zipfile, time
from collections import Counter, defaultdict
import cloudpickle, torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download

ARCH = 1
BASE = Path('/kaggle/input/datasets/kumaralpha1/a1-p3')
WEIGHTS, BPE = BASE / 'best.pt', BASE / 'tokenizer.json'
OUT = Path('/kaggle/working/phase6_a1_revised')
OUT.mkdir(exist_ok=True)
METHODS = ['lora', 'lorma', 'pfeiffer', 'houlsby']
SOURCE, TARGET, BATCH = 768, 192, 2
EPOCHS, PATIENCE, EVAL_STEPS = 8, 12, 250
SEED = 781
tok = Tokenizer.from_file(str(BPE))
PAD = tok.token_to_id('<pad>')
BOS = tok.token_to_id('<bos>')
EOS = tok.token_to_id('<eos>')
VOCAB = tok.get_vocab_size()

In [2]:
def read_rows(split):
    path = hf_hub_download('Exploration-Lab/CS-781-Capstone',
        f'Phase_6_Summ/{split.title()}/{split}.json', repo_type='dataset',
        token=os.environ['HF_TOKEN'])
    rows = json.loads(Path(path).read_text())
    if isinstance(rows, dict):
        rows = [dict(v, id=v.get('id', k)) if isinstance(v, dict) else {'id': k, 'document': v}
                for k, v in rows.items()]
    return rows

def prepare(split):
    data = []
    for row in read_rows(split):
        doc = tok.encode(row['document'], add_special_tokens=False).ids
        budget = SOURCE - 1
        if len(doc) > budget:
            quarter = budget // 4
            starts = [0, (len(doc)-quarter)//3, 2*(len(doc)-quarter)//3]
            doc = sum((doc[i:i+quarter] for i in starts), []) + doc[-(budget-3*quarter):]
        target = (tok.encode(row['summary'], add_special_tokens=False).ids[:TARGET-1] + [EOS]
                  if split == 'train' else [])
        data.append((row['id'], doc + [BOS], target))
    return data

rows = prepare('train')
ids = sorted({str(r[0]) for r in rows})
random.Random(SEED).shuffle(ids)
held = set(ids[:max(1, round(.1*len(ids)))])
train = [r for r in rows if str(r[0]) not in held]
dev = [r for r in rows if str(r[0]) in held]
test = prepare('test')
with (OUT/'data.pkl').open('wb') as f:
    pickle.dump((train,dev,test), f)
print(f'{len(train):,} train | {len(dev):,} dev | {len(test):,} test')

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

6,327 train | 703 dev | 100 test


In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        y = x.float()
        y = y * torch.rsqrt(y.square().mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['dim']
        self.heads = cfg['heads']
        self.head_dim = d // self.heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        inv = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('inv_freq', inv, persistent=False)

    def rope(self, x, positions):
        phase = positions.float().unsqueeze(-1) * self.inv_freq.float()
        cos = phase.cos().unsqueeze(1).to(x.dtype)
        sin = phase.sin().unsqueeze(1).to(x.dtype)
        even, odd = x[..., ::2], x[..., 1::2]
        return torch.stack((even * cos - odd * sin, even * sin + odd * cos), -1).flatten(-2)

    def forward(self, x, positions, mask=None, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.head_dim).unbind(2)
        q, k, v = (z.transpose(1, 2) for z in (q, k, v))
        q, k = self.rope(q, positions), self.rope(k, positions)
        old = 0
        if past is not None:
            old = past[0].size(2)
            k, v = torch.cat((past[0], k), 2), torch.cat((past[1], v), 2)
        allowed = None
        causal = mask is None and old == 0
        if mask is not None or old:
            key_idx = torch.arange(k.size(2), device=x.device)
            query_idx = old + torch.arange(t, device=x.device)
            allowed = key_idx[None, :] <= query_idx[:, None]
            allowed = allowed[None, None, :, :]
            if mask is not None:
                allowed = allowed & mask[:, None, None, :].bool()
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                          dropout_p=0.0, is_causal=causal)
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, d))
        return y, (k, v) if use_cache else None


class Bottleneck(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.down = nn.Linear(width, 32)
        self.up = nn.Linear(32, width)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x = x + self.up(F.relu(self.down(x)))
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg['dim'], cfg['ffn_dim']
        self.norm1, self.norm2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attention(cfg)
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x, positions, mask=None, past=None, use_cache=False):
        a, cache = self.attn(self.norm1(x), positions, mask, past, use_cache)
        x = x + a
        if hasattr(self, "att_adapter"): x = self.att_adapter(x)
        z = self.norm2(x)
        x = x + self.down(F.silu(self.gate(z)) * self.up(z))
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x, cache


class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['dim'])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg['layers'])])
        self.norm = RMSNorm(cfg['dim'])
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=0.02 / math.sqrt(2 * cfg['layers']))
            nn.init.normal_(block.down.weight, std=0.02 / math.sqrt(2 * cfg['layers']))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, ids, targets=None, attention_mask=None, past=None,
                use_cache=False, last_only=False):
        old = 0 if past is None else past[0][0].size(2)
        if attention_mask is None:
            positions = torch.arange(old, old + ids.size(1), device=ids.device)[None, :]
        else:
            positions = (attention_mask.long().cumsum(-1) - 1).clamp_min(0)[:, -ids.size(1):]
        x, caches = self.embed(ids), []
        for i, block in enumerate(self.blocks):
            if self.training and self.cfg.get('gradient_checkpointing', False):
                x = checkpoint(lambda z, m=block: m(z, positions)[0], x, use_reentrant=False)
            else:
                x, cache = block(x, positions, attention_mask,
                                 None if past is None else past[i], use_cache)
                if use_cache:
                    caches.append(cache)
        x = self.norm(x)
        if targets is not None:
            h, y = x.flatten(0, 1), targets.flatten()
            total = h.new_zeros((), dtype=torch.float32)
            for a in range(0, h.size(0), 256):
                def part(z, labels):
                    return F.cross_entropy(F.linear(z, self.embed.weight).float(), labels, reduction='sum')
                if self.training:
                    loss = checkpoint(part, h[a:a+256], y[a:a+256], use_reentrant=False)
                else:
                    loss = part(h[a:a+256], y[a:a+256])
                total = total + loss
            return total / y.numel()
        logits = F.linear(x[:, -1:] if last_only else x, self.embed.weight)
        return logits, caches if use_cache else None

In [4]:
class Adapt(nn.Module):
    def __init__(self, base, method):
        super().__init__()
        self.base, self.method = base, method
        if method == 'scale':
            self.scale = nn.Parameter(torch.ones(base.out_features))
        else:
            width = base.out_features if method == 'lorma' else base.in_features
            self.A = nn.Linear(width, 8, bias=False)
            self.B = nn.Linear(8, base.out_features, bias=False)
            nn.init.zeros_(self.B.weight)

    def forward(self, x):
        y = self.base(x)
        if self.method == 'scale':
            return y * self.scale.to(y.dtype)
        return y + self.B(self.A(y if self.method == 'lorma' else x))

In [5]:
def build(method):
    ck = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
    cfg = dict(ck['config'], gradient_checkpointing=False, pad_id=PAD)
    m = LegalLM(cfg)
    m.load_state_dict(ck['model'])
    m.requires_grad_(False)
    blocks = m.blocks
    width = cfg.get('dim',cfg.get('width'))
    if method in ('houlsby','pfeiffer'):
        for b in blocks:
            b.ff_adapter = Bottleneck(width)
            if method == 'houlsby': b.att_adapter = Bottleneck(width)
    else:
        for parent in list(m.modules()):
            for name, layer in list(parent.named_children()):
                if isinstance(layer, nn.Linear) and name in ('qkv','q','kv','out'):
                    setattr(parent, name, Adapt(layer, method))
    return m

In [6]:
def logits(m, batch):
    inp = pad([r[1]+r[2][:-1] for r in batch], PAD)
    lens = torch.tensor([len(r[1]) for r in batch], device=inp.device)
    y = pad([[-100]*(len(r[1])-1)+r[2] for r in batch], -100)
    x = m.embed(inp)
    pos = torch.arange(inp.size(1),device=inp.device)[None,:]
    for b in m.blocks: x = b(x,pos)[0]
    x = m.norm(x)
    keep = y != -100
    return F.linear(x[keep], m.embed.weight), y[keep]

In [7]:
def pad(rows, value):
    return nn.utils.rnn.pad_sequence([torch.tensor(r) for r in rows], batch_first=True,
                                     padding_value=value).cuda()

def batches(rows, seed=None):
    order = list(range(len(rows)))
    if seed is not None: random.Random(seed).shuffle(order)
    for start in range(0, len(order), 64):
        chunk = sorted(order[start:start+64], key=lambda i: len(rows[i][1])+len(rows[i][2]))
        for j in range(0,len(chunk),BATCH):
            yield [rows[i] for i in chunk[j:j+BATCH]]

def load_best(method):
    m = build(method)
    m.load_state_dict(torch.load(OUT/method/'best.pt',map_location='cpu',weights_only=True),strict=False)
    return m.cuda().eval()

def train_one(method):
    torch.manual_seed(SEED)
    with (OUT/'data.pkl').open('rb') as f: train, dev, _ = pickle.load(f)
    dest = OUT/method
    dest.mkdir(exist_ok=True)
    m = build(method).cuda()
    params = {k:p for k,p in m.named_parameters() if p.requires_grad}
    teacher = load_best('lora') if method == 'distill' else None
    lr = 2e-5 if method in ('partial','decoder','cross','distill') else 3e-4
    opt = torch.optim.AdamW(params.values(),lr=lr,fused=True)
    scaler = torch.amp.GradScaler('cuda')
    best, stale, step = float('inf'), 0, 0
    start = time.monotonic()
    print(method, 'trainable', sum(p.numel() for p in params.values()), flush=True)
    for epoch in range(EPOCHS):
        for j, batch in enumerate(batches(train, SEED+epoch), 1):
            m.train()
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                z,y = logits(m,batch)
                loss = F.cross_entropy(z.float(),y)
                if teacher is not None:
                    with torch.no_grad(): t,_ = logits(teacher,batch)
                    kd = F.kl_div((z.float()/2).log_softmax(-1),(t.float()/2).softmax(-1),reduction='batchmean')
                    loss = .5*loss + 2*kd
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params.values(),1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            if step % EVAL_STEPS and j != math.ceil(len(train)/BATCH): continue
            m.eval()
            total, count = torch.zeros((), device='cuda'), 0
            with torch.inference_mode(), torch.autocast('cuda',dtype=torch.float16):
                for v in batches(dev):
                    z,y = logits(m,v)
                    total += F.cross_entropy(z.float(),y,reduction='sum')
                    count += len(y)
            score = total.item()/count
            print(method, 'epoch',epoch+1,'step',step,'dev NLL',round(score,4),flush=True)
            if score < best:
                best, stale = score, 0
                torch.save({k:p.detach().half().cpu() for k,p in params.items()},dest/'best.tmp')
                os.replace(dest/'best.tmp',dest/'best.pt')
            else: stale += 1
            if stale >= PATIENCE: break
        if stale >= PATIENCE: break
    (dest/'done').write_text(json.dumps(dict(dev_nll=best,steps=step,seconds=time.monotonic()-start,
                    peak_GiB=torch.cuda.max_memory_allocated()/2**30)))

In [8]:
@torch.inference_mode()
@torch.autocast('cuda',dtype=torch.float16)
def generate(m,doc):
    ids = torch.tensor([doc],device='cuda')
    z,cache = m(ids,use_cache=True,last_only=True)
    z = z[:,-1]
    result, followers = [], defaultdict(set)
    for i in range(TARGET):
        z = z[:, :VOCAB].float()
        for special in (PAD,BOS):
            if special != EOS: z[:,special] = -torch.inf
        if i < 8: z[:,EOS] = -torch.inf
        if len(result)>=2:
            z[:,list(followers[tuple(result[-2:])])] = -torch.inf
        token = z.argmax(-1)
        n = token.item()
        if n == EOS: break
        if len(result)>=2: followers[tuple(result[-2:])].add(n)
        result.append(n)
        z,cache = m(token[:,None],past=cache,use_cache=True,last_only=True)
        z = z[:,-1]
    return result

def predict(method):
    m = load_best(method)
    tokenizer = Tokenizer.from_file(str(BPE))
    with (OUT/'data.pkl').open('rb') as f: _,_,test = pickle.load(f)
    rows = [dict(id=k,summary=tokenizer.decode(generate(m,d)).strip()) for k,d,_ in test]
    package(method,rows)

def package(method,rows):
    with zipfile.ZipFile(OUT/f'submission_{method}.zip','w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',json.dumps(rows,ensure_ascii=False))

In [9]:
def run_jobs(names, worker):
    jobs = {}
    queue = list(names)
    payload = OUT/'worker.pkl'
    payload.write_bytes(cloudpickle.dumps(worker))
    command = "import cloudpickle,sys,torch;torch.set_num_threads(1);cloudpickle.load(open(sys.argv[1],'rb'))(sys.argv[2])"
    while queue or jobs:
        for gpu in (0,1):
            if gpu in jobs:
                name,p = jobs[gpu]
                if p.poll() is None: continue
                p.wait()
                del jobs[gpu]
            ready = [n for n in queue if worker != train_one or n != 'distill' or (OUT/'lora'/'done').exists()]
            if ready:
                name = ready[0]
                queue.remove(name)
                print(f'GPU {gpu} | {name}',flush=True)
                jobs[gpu] = name,subprocess.Popen([sys.executable,'-c',command,str(payload),name],
                    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu)))
        if jobs: time.sleep(2)

In [10]:
run_jobs(METHODS, train_one)

GPU 0 | lora
GPU 1 | lorma
lora trainable 774144
lorma trainable 1032192
lorma epoch 1 step 250 dev NLL 2.521
lora epoch 1 step 250 dev NLL 2.5204
lorma epoch 1 step 500 dev NLL 2.5025
lora epoch 1 step 500 dev NLL 2.4807
lorma epoch 1 step 750 dev NLL 2.4666
lora epoch 1 step 750 dev NLL 2.451
lorma epoch 1 step 1000 dev NLL 2.4544
lora epoch 1 step 1000 dev NLL 2.4399
lorma epoch 1 step 1250 dev NLL 2.4387
lora epoch 1 step 1250 dev NLL 2.4314
lorma epoch 1 step 1500 dev NLL 2.4304
lora epoch 1 step 1500 dev NLL 2.4278
lorma epoch 1 step 1750 dev NLL 2.4227
lorma epoch 1 step 2000 dev NLL 2.4175
lora epoch 1 step 1750 dev NLL 2.422
lorma epoch 1 step 2250 dev NLL 2.4153
lora epoch 1 step 2000 dev NLL 2.4094
lorma epoch 1 step 2500 dev NLL 2.4154
lora epoch 1 step 2250 dev NLL 2.4062
lorma epoch 1 step 2750 dev NLL 2.4102
lora epoch 1 step 2500 dev NLL 2.4077
lorma epoch 1 step 3000 dev NLL 2.4099
lora epoch 1 step 2750 dev NLL 2.4007
lorma epoch 1 step 3164 dev NLL 2.4062
lorma epoch

In [11]:
run_jobs(METHODS, predict)

GPU 0 | lora
GPU 1 | lorma
GPU 1 | pfeiffer
GPU 0 | houlsby


In [12]:
def overlap(a,b):
    a,b = Counter(a.lower().split()),Counter(b.lower().split())
    return 2*sum((a&b).values())/max(1,sum(a.values())+sum(b.values()))

candidates = defaultdict(list)
for name in METHODS:
    with zipfile.ZipFile(OUT/f'submission_{name}.zip') as z:
        for row in json.loads(z.read('prediction.json')):
            candidates[row['id']].append(row['summary'])
ensemble = []
for key,_,_ in test:
    group = candidates[key]
    scores = [sum(overlap(s,t) for j,t in enumerate(group) if j != i) for i,s in enumerate(group)]
    ensemble.append(dict(id=key, summary=group[max(range(len(group)),key=scores.__getitem__)]))
package('ensemble',ensemble)
print(OUT)

/kaggle/working/phase6_a1_revised


## Notes